In [5]:
!pip install playwright pandas
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 19.6 MB/s eta 0:00:00
177 MiB [] 0% 359.8s177 MiB [] 0% 29.2s177 MiB [] 0% 16.4s177 MiB [] 0% 13.6s177 MiB [] 1% 6.5s177 MiB [] 1% 5.5s177 MiB [] 2% 3.8s177 MiB [] 3% 3.5s177 MiB [] 3% 3.9s177 MiB [] 3% 4.2s177 MiB [] 4% 4.6s177 MiB [] 4% 4.8s177 MiB [] 5% 4.1s177 MiB [] 6% 4.1s177 MiB [] 6% 4.2s177 MiB [] 6% 4.3s177 MiB [] 7% 4.0s177 MiB [] 8% 3.7s177 MiB [] 9% 3.4s177 MiB [] 10% 3.2s177 MiB [] 11% 3.0s177 MiB [] 12% 2.9s177 MiB [] 13% 2.7s177 MiB [] 14% 2.7s177 MiB [] 15% 2.6s177 MiB [] 16% 2.5s177 MiB [] 17% 2.4s177 MiB [] 18% 2.3s177 MiB [] 19% 2.2s177 MiB [] 20% 2.2s177 MiB [] 21% 2.1s177 MiB [] 22% 2.0s177 MiB [] 22% 2.1s177 MiB [] 22% 2.2s177 MiB [] 23% 2.1s177 MiB [] 24% 2.0s177 MiB [] 26% 1.9s177 MiB [] 27% 1.8s177 MiB [] 29% 1.7s177 MiB [] 30% 1.7s177 MiB [] 32% 1.6s177 MiB [] 33% 1.5s177 MiB [] 35% 1.4s177 MiB [] 36% 1.4s177 MiB [] 37% 1.3s177 MiB [] 39% 1.3s177 MiB [] 40% 1.2s177 MiB [] 40% 1.3s177 MiB [] 41% 1.3s177 

In [20]:
import os
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# Configurazione parametri
URL_TARGET = "https://osservatorioraccoltadifferenziata.it"
# ANNI = ["2020", "2021", "2022", "2023"]
ANNI = ["2020"]
# Elenco esatto delle regioni (devono corrispondere ai valori richiesti dal form del portale)
# REGIONI = [
#    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
#    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardia", "Marche",
#    "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", "Toscana",
#    "Trentino-Alto Adige", "Umbria", "Valle d'Aosta", "Veneto"
#]
REGIONI = ["Abruzzo"]

OUTPUT_DIR = "downloaded_csv"
FINAL_FILE = "raccolta_urbana_totale_2020_2023.csv"
TIMEOUT_SECONDI = 60  # Timeout aumentato a 30 secondi per dare tempo al server di rispondere

os.makedirs(OUTPUT_DIR, exist_ok=True)

def config_session():
    """Configura una sessione HTTP con politiche di retry automatico a livello di rete"""
    session = requests.Session()

    # Configura i tentativi di retry automatici (es. se la connessione cade o va in timeout)
    retries = Retry(
        total=3,                # Numero massimo di tentativi per singola URL
        backoff_factor=2,       # Aspetta 2s, poi 4s, poi 8s tra i tentativi falliti
        status_forcelist=[500, 502, 503, 504], # Riprova se il server restituisce questi errori
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    # Headers simulanti un vero browser per evitare blocchi immediati
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive"
    })
    return session

def download_all_data():
    all_files = []
    session = config_session()

    for anno in ANNI:
        for regione in REGIONI:
            print(f"Inizio download per: Anno {anno} - Regione {regione}...")

            # Controllo se il file è già stato scaricato in una sessione precedente per evitare di rifarlo
            filename = f"dati_{regione}_{anno}.csv".lower().replace(" ", "_").replace("'", "")
            filepath = os.path.join(OUTPUT_DIR, filename)

            if os.path.exists(filepath):
                print(f"File già presente localmente, salto: {filename}")
                all_files.append(filepath)
                continue

            try:
                # 1. Recupera la pagina del form (con timeout esplicito)
                response = session.get(URL_TARGET, timeout=TIMEOUT_SECONDI)
                soup = BeautifulSoup(response.text, 'html.parser')

                # 2. Struttura del payload
                payload = {
                    'anno': anno,
                    'regione': regione,
                    'submit': 'Ricerca'
                }

                # Recupera campi nascosti generati dinamicamente dalla piattaforma
                for hidden_input in soup.find_all("input", type="hidden"):
                    if hidden_input.get("name"):
                        payload[hidden_input.get("name")] = hidden_input.get("value", "")

                # 3. Richiesta POST per i risultati
                post_response = session.post(URL_TARGET, data=payload, timeout=TIMEOUT_SECONDI)
                post_soup = BeautifulSoup(post_response.text, 'html.parser')

                # 4. Individuazione link di download
                csv_link = None
                for a_tag in post_soup.find_all("a", href=True):
                    if "csv" in a_tag["href"].lower() or "download" in a_tag["href"].lower():
                        csv_link = a_tag["href"]
                        break

                if not csv_link:
                    download_form = post_soup.find("form", id=lambda x: x and "download" in x.lower())
                    if download_form:
                        action = download_form.get("action", URL_TARGET)
                        download_payload = {hid.get("name"): hid.get("value", "") for hid in download_form.find_all("input")}
                        csv_response = session.post(action, data=download_payload, timeout=TIMEOUT_SECONDI)
                    else:
                        raise Exception("Pulsante CSV non rintracciato nella pagina di risposta.")
                else:
                    if not csv_link.startswith("http"):
                        csv_link = requests.compat.urljoin(URL_TARGET, csv_link)
                    csv_response = session.get(csv_link, timeout=TIMEOUT_SECONDI)

                # 5. Scrittura del file scaricato
                if csv_response.status_code == 200:
                    with open(filepath, "wb") as f:
                        f.write(csv_response.content)
                    all_files.append(filepath)
                    print(f"Salvato con successo: {filename}")
                else:
                    print(f"Errore di download per {regione} ({anno}). Status: {csv_response.status_code}")

            except requests.exceptions.RequestException as req_err:
                print(f"Errore di rete/timeout per {regione} - Anno {anno}: {req_err}")
            except Exception as e:
                print(f"Errore generico per {regione} - Anno {anno}: {e}")

            # Pausa di cortesia aumentata a 3 secondi per evitare di attivare i sistemi anti-flood
            time.sleep(3)

    return all_files

def merge_csv_files(files_list):
    if not files_list:
        print("\nNessun file CSV valido a disposizione per l'unione.")
        return

    print("\nInizio unione dei file CSV...")
    combined_df = pd.DataFrame()

    for file in files_list:
        try:
            # Adeguato per leggere i formati standard (prova sep="," se i dati risultano disallineati)
            df = pd.read_csv(file, sep=";")
            combined_df = pd.concat([combined_df, df], ignore_index=True)
        except Exception as e:
            print(f"Impossibile leggere il file {file}: {e}")

    combined_df.to_csv(FINAL_FILE, index=False, sep=";", encoding="utf-8-sig")
    print(f"\nOperazione completata! Archivio unico disponibile in: {FINAL_FILE}")

# Esecuzione nel notebook
scaricati = download_all_data()
merge_csv_files(scaricati)


Inizio download per: Anno 2020 - Regione Abruzzo...


Errore di rete/timeout per Abruzzo - Anno 2020: HTTPSConnectionPool(host='osservatorioraccoltadifferenziata.it', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f053d8aca40>, 'Connection to osservatorioraccoltadifferenziata.it timed out. (connect timeout=60)'))

Nessun file CSV valido a disposizione per l'unione.


In [21]:
import socket
import requests

# Test 1: risoluzione DNS
try:
    ip = socket.gethostbyname("osservatorioraccoltadifferenziata.it")
    print("DNS OK, IP:", ip)
except Exception as e:
    print("DNS FALLITO:", e)

# Test 2: connessione TCP pura sulla porta 443
try:
    s = socket.create_connection(("osservatorioraccoltadifferenziata.it", 443), timeout=10)
    print("TCP connect OK")
    s.close()
except Exception as e:
    print("TCP connect FALLITO:", e)

# Test 3: richiesta HTTP semplice con timeout basso
try:
    r = requests.get("https://osservatorioraccoltadifferenziata.it", timeout=10)
    print("HTTP status:", r.status_code)
except Exception as e:
    print("HTTP FALLITO:", e)

DNS OK, IP: 91.226.108.23
TCP connect FALLITO: timed out
HTTP FALLITO: HTTPSConnectionPool(host='osservatorioraccoltadifferenziata.it', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f056504d910>, 'Connection to osservatorioraccoltadifferenziata.it timed out. (connect timeout=10)'))
